# HW5. Короткий запуск на GPU

Выполняй на одном CUDA GPU в Docker-окружении из этого проекта. Все методы импортируются из .py. Этот notebook проверяет настоящую генерацию, callback награды, несколько шагов GRPO, изменение LoRA-весов, сохранение адаптера и расход памяти. Он не запускает BFCL и не является одним из трёх основных экспериментов.

Нужны доступ к Hugging Face и свободная GPU-память. Образ и SSH-туннель описаны в README.md. Данные и веса загружаются здесь на сервере, а не на локальной машине.

In [ ]:
import json
import subprocess
from datetime import datetime
from pathlib import Path
from pprint import pprint

import torch

from download_data import download_data
from prepare import prepare_data
from train import run_training

subprocess.run(['nvidia-smi'], check=True)
print('PyTorch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())

## 1. Небольшой настоящий ToolACE

Для короткого запуска достаточно первых 200 диалогов. Разделение выполняется по диалогам до нарезки. В реальном обучении используются только ToolACE и сгенерированные текущей моделью ответы: подставленные локальные ответы сюда не передаются.

In [ ]:
input_path = Path('data/raw/data.json')
if not input_path.exists():
    download_data()
report = prepare_data(input_path, 'data/prepared', limit=200)
print('Train-примеров:', report['train_examples'])
print('Validation-примеров:', report['validation_examples'])
print('Исключённых диалогов:', len(report['excluded_dialogues']))
pprint(report['excluded_dialogues'][:5])

## 2. Конфигурация проверки

configs/smoke.json задаёт пять optimizer-шагов, четыре генерации на запрос, максимум 512 новых токенов и LoRA rank 8 на q_proj/v_proj. GRPO использует beta=0, чтобы не загружать вторую модель. Генерация идёт через Transformers, без отдельного экземпляра vLLM. Микробатч 1 и накопление 4 дают полный набор из четырёх ответов на запрос.

Награда: exact + 0.2 × functions + 0.05 × format. Это стартовая смесь для проверки механизма, а не выбранная лучшая конфигурация. Ответы, достигшие лимита без EOS, получают ноль и исключаются из loss. Prompt длиннее 2048 токенов исключается целиком с подсчётом; он не обрезается скрыто.

Каждая попытка использует отдельную папку. Уже записанные результаты train.py не перезаписывает.

In [ ]:
config_path = Path('configs/smoke.json')
settings = json.loads(config_path.read_text())
pprint(settings)
output_dir = Path('results') / f"smoke_{datetime.now():%Y%m%d_%H%M%S}"
print('Результаты:', output_dir)

## 3. Настоящее обучение

Здесь впервые загружаются веса исходной Qwen закреплённой ревизии. Обучаются только адаптеры. run_training сохраняет конфигурацию, реальные генерации с компонентами награды, историю метрик, адаптер и summary.json. Расход памяти измеряется средствами PyTorch на фактическом запуске.

In [ ]:
summary = run_training(
    data_path='data/prepared/train.jsonl',
    config_path=config_path,
    output_dir=output_dir,
)
pprint(summary)

## 4. Посмотреть реальную генерацию и награды

generations.jsonl содержит именно генерации модели. Для каждой сохранены шаг, группа ответов на один запрос, эталон, компоненты награды, итоговая награда, длина, признак обрезания и выделенная GPU-память. history.json содержит метрики optimizer-шагов.

In [ ]:
generations = []
for line in (output_dir / 'generations.jsonl').read_text().splitlines():
    generations.append(json.loads(line))
for generation in generations[:4]:
    print('Шаг:', generation['step'], 'группа:', generation['group'])
    print(generation['response'])
    print('Компоненты:', generation['scores'], 'награда:', generation['reward'])
    print('Обрезан:', generation['truncated'])
print('История обучения:')
pprint(json.loads((output_dir / 'history.json').read_text()))

## 5. Что считать успешной проверкой

Должны завершиться все optimizer-шаги; метрики должны быть конечными; реальные LoRA-параметры должны измениться. parameter_delta_l2 измеряет евклидову норму изменения обучаемых весов. peak_allocated_gib и peak_reserved_gib — максимальная выделенная и зарезервированная PyTorch память в GiB (байты / 1024³), включая загрузку модели.

Если все ответы в каждой группе получили одинаковую награду, advantage равен нулю и обучение может не изменить веса. В этом случае notebook не объявит успех: посмотри generations.jsonl, число обрезанных ответов и groups_with_reward_variation. Для следующей попытки увеличь число шагов или генераций, уточни награду либо лимит ответа по реальным выводам и создай новую папку результатов. Завершение процесса само по себе не доказывает обновление модели.

Сохранён adapter, а не полная объединённая модель. Для будущего BFCL нужен отдельный экспорт через train.export_model. Baseline и полноценные эксперименты находятся в experiments.ipynb.

In [ ]:
print('Пик выделенной памяти, GiB:', summary['peak_allocated_gib'])
print('Пик зарезервированной памяти, GiB:', summary['peak_reserved_gib'])
print('Изменение обучаемых весов, L2:', summary['parameter_delta_l2'])
print('Групп с различными наградами:', summary['groups_with_reward_variation'])
assert summary['optimizer_steps'] == settings['max_steps']
assert summary['finite_training_metrics']
assert summary['generated_completions'] > 0
assert summary['peak_allocated_gib'] > 0
assert summary['changed_parameter_tensors'] > 0, 'Веса не изменились; проверь реальные награды'
print('Короткий запуск прошёл: генерация, награда, optimizer, адаптер и GPU-память проверены.')